In [ ]:
# ============================================================================
#  STEP 1:  Install All Dependencies
# ============================================================================

print("📦 Installing dependencies...")

# Core dependencies
!pip install txtai[pipeline] chromadb sentence-transformers -q
!pip install gradio -q
!pip install bitsandbytes scipy nltk -q

# Install transformers and compatible huggingface-hub
!pip install 'transformers>=4.40.0' -q
!pip install 'huggingface-hub>=0.30.0,<1.0' -q

# Install torch and accelerate
!pip install torch accelerate -q

# Datasets with compatible version
!pip install 'datasets>=2.0.0' -q

print("✅ All dependencies installed with compatible versions!")

# Verify versions
print("\n📋 Checking key package versions:")
import transformers
import huggingface_hub
print(f"  transformers: {transformers.__version__}")
print(f"  huggingface_hub: {huggingface_hub.__version__}")


📦 Installing dependencies...
✅ All dependencies installed with compatible versions!

📋 Checking key package versions:
  transformers: 4.51.0
  huggingface_hub: 0.36.0


In [ ]:

# ============================================================================
# STEP 2:  Mount Google Drive & Import Libraries
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

from txtai.pipeline import FileToHTML, HTMLToMarkdown, Segmentation
from txtai import Embeddings
import chromadb
from chromadb.utils import embedding_functions
import gradio as gr
from pathlib import Path
import json
import uuid
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from typing import Dict, List

# Download required NLTK data
print("📥 Downloading NLTK resources...")
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('cmudict', quiet=True)

print("✅ Libraries imported successfully!")
print("✅ NLTK resources downloaded!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Downloading NLTK resources...
✅ Libraries imported successfully!
✅ NLTK resources downloaded!


In [ ]:
# ============================================================================
# STEP 3: Load and Process PDF Documents
# ============================================================================
print("Processing PDF documents...")

# Specify your PDF path here
pdfs = ["/content/drive/MyDrive/White-Water-Canyon-2016.pdf"]

# Initialize document processors with better settings
tohtml = FileToHTML(backend="docling")
tomd = HTMLToMarkdown()
segment = Segmentation(paragraphs=True, minlength=100, sentences=True)

# Sanitization function for chunks
def sanitize_chunk(text):
    import re
    patterns = [
        r"\[/?INST\]",
        r"\[/?prompt\]",
        r"^(Human:|User:|Assistant:|Q:|A:).*?$",
        r"^Please clarify.*?$",
        r"^Summarize.*?$"
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.MULTILINE)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

# Process documents into sanitized chunks
chunks = []
for path in pdfs:
    print(f"Processing {path}...")
    html = tohtml(path)
    md = tomd(html)
    for i, para in enumerate(segment(md)):
        if len(para.strip()) < 20:
            continue
        tags_string = json.dumps({"source": path})
        clean_para = sanitize_chunk(para)
        if clean_para:
            chunks.append((f"{path}-{i}", clean_para, tags_string))

print(f"Created {len(chunks)} sanitized text chunks.")

# Create hybrid embeddings
print("Creating hybrid search index...")
hybrid = Embeddings(hybrid=True, dense=True, content=True)
hybrid.index(chunks)
print(f"Hybrid search index created with {len(chunks)} chunks.")

Processing PDF documents...
Processing /content/drive/MyDrive/White-Water-Canyon-2016.pdf...


[INFO] 2025-11-15 17:57:06,908 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-11-15 17:57:06,933 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-11-15 17:57:06,935 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-11-15 17:57:07,051 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-11-15 17:57:07,058 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-11-15 17:57:07,061 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-11-15 17:57:07,122 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-11-15 17:57:07,177 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

Created 729 sanitized text chunks.
Creating hybrid search index...
Hybrid search index created with 729 chunks.


In [ ]:

# ============================================================================
#  STEP 4: Initialize ChromaDB Vector Database
# ============================================================================
print("🗄️ Initializing ChromaDB vector database...")

# Prepare documents and metadata
documents = [chunk[1] for chunk in chunks]
metadatas = [json.loads(chunk[2]) for chunk in chunks]

# Initialize ChromaDB client
chroma_client = chromadb.Client()

# Create sentence transformer embedding function
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create or reset collection
collection_name = "wwc_knowledge_base"
try:
    chroma_client.delete_collection(name=collection_name)
    print("  Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=sentence_transformer_ef
)

# Add documents to collection
ids = [str(uuid.uuid4()) for _ in documents]
collection.add(documents=documents, metadatas=metadatas, ids=ids)

print(f"✅ ChromaDB ready with {collection.count()} documents")

🗄️ Initializing ChromaDB vector database...
✅ ChromaDB ready with 729 documents


In [ ]:

# ============================================================================
# STEP 5: CLEAR GPU MEMORY
# ============================================================================
import torch
import gc

print("🧹 Clearing GPU memory...")

# Clear any existing models from GPU
if 'llm' in globals():
    del llm
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer

# Force garbage collection
gc.collect()

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"✅ GPU memory cleared!")
    print(f"GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB allocated")
else:
    print("⚠️ CUDA not available")


🧹 Clearing GPU memory...
✅ GPU memory cleared!
GPU Memory: 0.46 GB allocated


In [ ]:
# ============================================================================
# STEP 6: Initialize Language Model
# ============================================================================
import torch
from typing import Dict, List
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("🤖 Loading Language Model...")

class HuggingFaceLLM:
    """Wrapper for Hugging Face language models"""

    def __init__(self, model_name: str = "Qwen/Qwen2.5-3B-Instruct"):
        print(f"  Loading {model_name}...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"  Using device: {self.device}")

        # Use 4-bit quantization (more memory efficient than 8-bit)
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        print("  Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print("  Loading model (this may take 2-3 minutes)...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            max_memory={0: "13GB"}  # Limit GPU memory usage
        )

        print(f"  ✅ Model loaded successfully!")
        if torch.cuda.is_available():
            print(f"  GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

    def generate(self, prompt: str, max_length: int = 300) -> str:
        """Generate text response from prompt"""
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            max_length=2048,
            truncation=True,
            padding=True
        )

        input_device = next(self.model.parameters()).device
        inputs = {k: v.to(input_device) for k, v in inputs.items()}

        with torch.no_grad():  # Save memory during inference
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,  # More natural responses
                top_p=0.9,        # Better diversity
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        # Decode only the NEW tokens (generated response)
        input_length = inputs['input_ids'].shape[1]
        generated_tokens = outputs[0][input_length:]
        response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        return response

# Initialize with 3B model (much more memory efficient than 7B)
llm = HuggingFaceLLM(model_name="Qwen/Qwen2.5-3B-Instruct")
print("\n✅ Language model ready!")


🤖 Loading Language Model...
  Loading Qwen/Qwen2.5-3B-Instruct...
  Using device: cuda
  Loading tokenizer...
  Loading model (this may take 2-3 minutes)...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  ✅ Model loaded successfully!
  GPU Memory: 2.38 GB

✅ Language model ready!


In [ ]:
# ============================================================================
# STEP 7: Define RAG Pipeline Function
# ============================================================================
def generate_rag_response(query: str, top_k: int = 3) -> Dict:
    """
    Complete RAG pipeline that retrieves relevant documents and generates response

    Args:
        query: User's question
        top_k: Number of relevant documents to retrieve

    Returns:
        Dictionary containing response and source documents
    """
    # Step 1: Retrieve relevant documents using hybrid search
    txtai_results = hybrid.search(query, top_k)
    txtai_retrieved_docs = [doc['text'] for doc in txtai_results]
    txtai_scores = [doc['score'] for doc in txtai_results]

    # Step 2: Build context from retrieved documents
    def sanitize_chunk(text):
        import re
        return re.sub(r"[\[\]]/?INST\]|[\[\]]/?prompt\]|^Q:.*$|^A:.*$", "", text, flags=re.MULTILINE)

    context = "\n".join(sanitize_chunk(doc) for doc in txtai_retrieved_docs)

    # Step 3: Create improved prompt for the language model with better engineering
    prompt = f"""<|im_start|>system
You are an expert operations assistant for the White Water Canyon ride at an amusement park. Your role is to provide accurate, actionable information to ride operators based on the official operations manual.

CRITICAL RULES:
- Answer ONLY using information from the provided context
- If information is not in the context, clearly state "This information is not covered in the manual, ask Park Operations."
- For safety procedures, be extremely precise and include all steps
- Use clear, professional language appropriate for trained operators
- When multiple procedures exist, present them as numbered steps or bullet points in a logical order
- Always prioritize safety-critical information first
<|im_end|>

<|im_start|>user
MANUAL CONTEXT:
{context}

OPERATOR QUESTION:
{query}

Provide a clear, complete response. If the question involves emergency procedures or safety protocols, include all relevant steps and warnings.
<|im_end|>

<|im_start|>assistant"""

    # Step 4: Generate response using LLM
    response = llm.generate(prompt, max_length=250)

    return {
        'response': response,
        'retrieved_docs': txtai_retrieved_docs,
        'scores': txtai_scores
    }

print("✅ RAG pipeline function defined!")

✅ RAG pipeline function defined!


In [ ]:
# ============================================================================
# STEP 8: Create Chatbot Interface (UPDATED)
# ============================================================================

def chat_with_rag(message: str, history: List, show_sources: bool = False) -> str:
    try:
        print(f"Received message: {message}")
        result = generate_rag_response(message, top_k=3)
        print(f"RAG result generated successfully")

        response = result['response']

        if show_sources:
            response += "\n\n---\n\n📚 **Sources from Manual:**\n"
            if result.get('retrieved_docs'):
                for i, (doc, score) in enumerate(zip(result['retrieved_docs'],
                                                      result['scores']), 1):
                    doc_preview = doc[:200] + "..." if len(doc) > 200 else doc
                    response += f"\n**{i}.** *(Relevance: {score:.2%})*\n{doc_preview}\n"
            else:
                response += "No sources found for this query."

        return response

    except Exception as e:
        print(f"Error in chat_with_rag: {e}")
        return f"⚠️ **Error:** An internal error occurred: {str(e)}\n\nPlease try rephrasing your question or contact support."

# Example questions
example_questions = [
    ["What number do I call whenever rides or attractions close?"],
    ["What should I do if there's a man overboard?"],
    ["How do I evacuate Lake 1?"],
    ["What are the height restrictions for guests?"],
    ["What is the procedure for a Code 2 situation?"],
    ["How do I use the backup gate release valve?"],
    ["What are the opening procedures for Tower 1?"],
    ["What should I do if the motor is falling off?"],
    ["How do I handle a boat jam in Lake 2?"],
    ["What are the safety equipment locations?"]
]

# Strict black-and-white CSS
custom_css = """
/* Force entire app background to black */
body, .gradio-container, .block, .wrap, #chatbot {
    background-color: #000000 !important;
    color: #ffffff !important;
}

/* User messages */
.user {
    background-color: #000000 !important;
    color: #ffffff !important;
}

/* Bot messages */
.bot {
    background-color: #000000 !important;
    color: #ffffff !important;
}

/* Markdown elements */
p, span, strong, em, li, h1, h2, h3, h4, h5, h6 {
    color: #ffffff !important;
}

/* Links */
a {
    color: #ffffff !important;
    text-decoration: underline !important;
}

/* Code blocks */
code, pre {
    color: #ffffff !important;
    background-color: #000000 !important;
}

/* Example questions */
.examples, .examples * {
    background-color: #000000 !important;
    color: #ffffff !important;
}

/* Example buttons */
.examples button {
    background-color: #000000 !important;
    color: #ffffff !important;
    border: 1px solid #ffffff !important;
}

/* Buttons (Send, Clear Chat) */
button {
    background-color: #000000 !important;
    color: #ffffff !important;
    border: 1px solid #ffffff !important;
}

/* Description text */
.description, .description * {
    color: #ffffff !important;
}

/* Chatbot container */
#chatbot {
    background-color: #000000 !important;
    height: 600px;
}
"""

print("🚀 Creating chatbot interface...")

# Create the Gradio interface WITHOUT theme overrides
with gr.Blocks(
    css=custom_css,
    title="🌊 White Water Canyon Operations Assistant"
) as demo:

    gr.Markdown("""
    # 🌊 White Water Canyon Operations Assistant

    **Welcome to the WWC Operations Manual Chatbot!**

    Ask questions about:
    - Ride operations and procedures
    - Safety protocols and emergency responses
    - Equipment locations and usage
    - Code procedures (Code 1, 2, 3, etc.)
    - Evacuation procedures
    - Station operations

    ⚠️ **Important:** This chatbot provides information from the operations manual.
    Always follow official protocols and consult supervisors for critical decisions.
    """)

    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                height=600,
                show_label=False,
                container=True,
                type="tuples"
            )
        with gr.Column(scale=1):
            show_sources_checkbox = gr.Checkbox(
                label="Show Sources from Manual",
                value=False,
                info="Enable to see source documents with each response"
            )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask about procedures, codes, safety protocols...",
            container=False,
            scale=7,
            show_label=False
        )
        submit_btn = gr.Button("Send", scale=1)

    gr.Examples(
        examples=example_questions,
        inputs=msg,
        label="Example Questions"
    )

    def respond(message, chat_history, show_sources):
        bot_message = chat_with_rag(message, chat_history, show_sources)
        chat_history.append((message, bot_message))
        return "", chat_history

    msg.submit(respond, [msg, chatbot, show_sources_checkbox], [msg, chatbot])
    submit_btn.click(respond, [msg, chatbot, show_sources_checkbox], [msg, chatbot])

    clear = gr.Button("Clear Chat")
    clear.click(lambda: None, None, chatbot, queue=False)

print("✅ Chatbot interface created!")
print("\n" + "="*70)
print("🎉 READY TO LAUNCH!")
print("="*70)


🚀 Creating chatbot interface...
✅ Chatbot interface created!

🎉 READY TO LAUNCH!


/tmp/ipython-input-639070940.py:139: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  chatbot = gr.Chatbot(


In [ ]:
# ============================================================================
# STEP 9: Launch the Gradio Chatbot
# ============================================================================
demo.launch(
    share=True,
    debug=True,
    show_error=True
)

print("\n✅ Chatbot is now running!")
print("📱 Use the link above to access your chatbot")
print("🔗 The 'share=True' parameter creates a public link valid for 72 hours")

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b560e3d05d3b66a58a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Received message: What should I do if there's a man overboard?
RAG result generated successfully
Received message: where is the jigaloo ride?
RAG result generated successfully
Received message: what the 7-4 operator is?
RAG result generated successfully
Received message: what do I do if there is a power outage?
RAG result generated successfully
Received message: how to knock a passenger off the ride?
RAG result generated successfully
Received message: what is the human condition?
RAG result generated successfully
Received message: what to do in inclement weather?
RAG result generated successfully


In [ ]:
# ============================================================================
#  Define and Launch the Gradio Chatbot
# ============================================================================

#def chatbot_interface(query, history):
    #"""
   # Gradio chatbot interface function.
   # """
   # response_data = generate_rag_response(query)
   # full_response = response_data['response']

    # Display source documents in a user-friendly format
    #sources = "\n\n---\n### Sources:\n"
   # for i, doc in enumerate(response_data['retrieved_docs']):
        sources += f"- Document {i+1}: {doc[:150]}...\n"

    # Combine response and sources
   # return full_response + sources

# Create the Gradio Interface
#demo = gr.ChatInterface(
   # chatbot_interface,
    #title="White Water Canyon Operations Assistant",
   # description="Ask any question about the White Water Canyon ride manual. I will retrieve relevant information and answer based on the manual's content."
#)

# Launch the Chatbot
#demo.launch(
   # share=True,
   # debug=True,
   # show_error=True
#)

#print("\n✅ Chatbot is now running!")
#print("\U0001F4F1 Use the link above to access your chatbot")
#print("\U0001F517 The 'share=True' parameter creates a public link valid for 72 hours")